In [1]:
import itertools
import math
import cmath

from qiskit import QuantumCircuit, QuantumRegister
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
import ffsim
import pyscf
from pyscf import gto, scf, cc
import json
from ffsim.linalg import givens_decomposition
from qiskit_ibm_runtime import  SamplerV2 as Sampler
from qiskit.circuit.library import XXPlusYYGate, RYGate, RZZGate, CPhaseGate

from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import CouplingMap

## Molecule definition

In [2]:
def generate_hchain_geometry(natoms: int, atomic_distance: float = 0.7) -> str:
    """Returns a linear Hydrogen chain geometry for use in PySCF molecule construction.
    
    Args:
        natoms: Number of Hydrogen atoms in the chain.
        atomic_distance: Equal spacing between Hydrogen atoms.
    """
    return "; ".join([f"H 0 0 {i * atomic_distance}" for i in range(natoms)])

In [3]:
geom = generate_hchain_geometry(natoms=8)
print(geom)

mol = gto.Mole(atom = geom, charge = 0, basis = 'sto3g')
mol.build()
mf = scf.RHF(mol)
mf.verbose = 0
mf.kernel()
cc_ = cc.CCSD(mf).run()
N = mol.nao_nr() * 2

H 0 0 0.0; H 0 0 0.7; H 0 0 1.4; H 0 0 2.0999999999999996; H 0 0 2.8; H 0 0 3.5; H 0 0 4.199999999999999; H 0 0 4.8999999999999995
E(CCSD) = -4.049928261732551  E_corr = -0.07792535779860998


## FFSIM LUCJ

Following https://quantum.cloud.ibm.com/docs/en/tutorials/sample-based-quantum-diagonalization.

In [4]:
# Define active space
n_frozen = 0
active_space = range(n_frozen, mol.nao_nr())


# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Compute exact energy using FCI
reference_energy = cas.run().e_tot

print(f"norb = {norb}")
print(f"nelec = {nelec}")

converged SCF energy = -3.97200290393394
CASCI E = -4.05042857904067  E(CI) = -14.4395811689394  S^2 = 0.0000000
norb = 8
nelec = (4, 4)


In [5]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

E(CCSD) = -4.049928261732551  E_corr = -0.07792535779860998


In [6]:
# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = [(p, p) for p in range(norb)]  # None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

# Initialize backend
coupling_map = CouplingMap.from_heavy_hex(3)
backend = GenericBackendV2(
    coupling_map.size(),
    coupling_map=coupling_map,
    basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()

/Users/thomasvancamp/acres/.venv/lib/python3.14/site-packages/ffsim/qiskit/lucj_pass_manager.py:160: UserWarning: Backend cannot accommodate pairs_ab=[(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6), (7, 7)].
Removing interaction (7, 7) from the end.
  warnings.warn(
/Users/thomasvancamp/acres/.venv/lib/python3.14/site-packages/ffsim/qiskit/lucj_pass_manager.py:160: UserWarning: Backend cannot accommodate pairs_ab=[(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6)].
Removing interaction (6, 6) from the end.
  warnings.warn(
/Users/thomasvancamp/acres/.venv/lib/python3.14/site-packages/ffsim/qiskit/lucj_pass_manager.py:160: UserWarning: Backend cannot accommodate pairs_ab=[(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 5)].
Removing interaction (5, 5) from the end.
  warnings.warn(
/Users/thomasvancamp/acres/.venv/lib/python3.14/site-packages/ffsim/qiskit/lucj_pass_manager.py:160: UserWarning: Backend cannot accommodate pairs_ab=[(0, 0), (1, 1), (2, 2), (3, 3), (4, 4)].
Removin

In [7]:
isa_circuit = pass_manager.run(circuit)
isa_circuit.draw(fold=-1)

┌──────────────────────────┐                                                        ┌──────────────────────────┐                                                          ┌─────────────────────────┐                                                                                                                                                                                                                                 ┌──────────────────────────┐┌──────────────────────────┐                                                                                                                                                                                                                                                                                                                                                                 ┌───────────────────────────┐                                                                                                                                                                                                                      ┌──────────────────────────┐┌──────────────────────────┐                                                                                                                                                                                                                                                                                    ┌──────────────────────────┐┌──────────────────────────┐                                                                                                                                                                                                                                                                                  ┌──────────────────────────┐┌─────────────────────────┐                                                                                                                                                                                                                                                                                          ┌───────────────────────────┐       ┌────────────┐                                                                                                           ░                                        ┌─┐      
     q_13 -> 0 ───────────────────────────────────────────────────────────┤0                         ├────────────────────────────────────────────────────────┤1                         ├──────────────────────────────────────────────────────────┤0                        ├─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤1                         ├┤0                         ├───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■─────────────────────────────────────────────────■───────────────────────────────────┤0                          ├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤1                         ├┤0                         ├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤0                         ├┤1                         ├───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [8]:
isa_circuit.count_ops()

OrderedDict([('xx_plus_yy', 88),
             ('p', 16),
             ('measure', 16),
             ('cp', 15),
             ('x', 8),
             ('swap', 2),
             ('barrier', 1)])

## Custom LUCJ

In [9]:
def orbital_rotation(qc_lucj, orbital_rotation, N):
    givens_rotations, phase_shifts = givens_decomposition(orbital_rotation)
    for c, s, i, j in givens_rotations:
        # print(i, j)
        qc_lucj.append(XXPlusYYGate(theta = 2 * math.acos(c), beta = cmath.phase(s) - 0.5 * math.pi), [i, j])
        qc_lucj.append(XXPlusYYGate(theta = 2 * math.acos(c), beta = cmath.phase(s) - 0.5 * math.pi), [i + mol.nao_nr(), j + mol.nao_nr()])

    for i, phase_shift in enumerate(phase_shifts):
        qc_lucj.p(cmath.phase(phase_shift), i)
        qc_lucj.p(cmath.phase(phase_shift), i + mol.nao_nr())

    qc_lucj.barrier()

In [10]:
def J_op(qc_lucj, diag_mat_aa, diag_mat_ab, diag_mat_bb, time, norb):
    for sigma, this_mat in enumerate([diag_mat_aa, diag_mat_bb]):
        if this_mat is None:
            print('______________________')
            print('_________NONE_________')
            print('______________________')
        if this_mat is not None:
            for i in range(norb):
                if this_mat[i, i]:
                    # print(i + sigma * norb)
                    qc_lucj.p(-0.5 * this_mat[i, i] * time, i + sigma * norb)
            for i, j in itertools.combinations(range(norb), 2):
                if this_mat[i, j]:
                    # print(i + sigma * norb, j + sigma * norb)
                    qc_lucj.cp(-this_mat[i, j] * time, i + sigma * norb, j + sigma * norb)
    # qc_lucj.barrier()
    if diag_mat_ab is not None:
        for i in range(norb):
            if diag_mat_ab[i, i]:
                qc_lucj.cp(-diag_mat_ab[i, i] * time, i, i + norb)
                
        for i, j in itertools.combinations(range(norb), 2):
            
            if diag_mat_ab[i, j]:
                qc_lucj.cp(-diag_mat_ab[i, j] * time, i, j + norb)
                
            if diag_mat_ab[j, i]:
                qc_lucj.cp(-diag_mat_ab[j, i] * time, j, i + norb)
                

    qc_lucj.barrier()

In [11]:
qc_lucj = QuantumCircuit(N)
t1, t2 = cc_.t1, cc_.t2

norb = mol.nao_nr()
pairs_aa = [(p, p + 1) for p in range(norb - 1)]   # same-spin line
pairs_ab = [(p, p) for p in range(norb)]           # opposite-spin on-site
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=cc_.t2, t1=cc_.t1,
    n_reps=2,
    interaction_pairs=(pairs_aa, pairs_ab),
)

orbital_rotations = ucj_op.orbital_rotations
diag_mats = ucj_op.diag_coulomb_mats
final_orbital_rotation = ucj_op.final_orbital_rotation
for i in range(mol.nelectron // 2):
    qc_lucj.x(i)
    qc_lucj.x(i + mol.nao_nr())
for orb_rot, (diag_mat_aa, diag_mat_ab) in zip(orbital_rotations, diag_mats):
    orbital_rotation(qc_lucj, orb_rot.T.conj(), N)
    J_op(qc_lucj, diag_mat_aa, diag_mat_ab, diag_mat_aa, time=-1, norb=mol.nao_nr())
    orbital_rotation(qc_lucj, orb_rot, N)
if ucj_op.final_orbital_rotation is not None:
    orbital_rotation(qc_lucj, ucj_op.final_orbital_rotation, N)

In [12]:
qc_lucj_raw = QuantumCircuit(N)

for i in range(mol.nelectron // 2):
    qc_lucj_raw.x(i)
    qc_lucj_raw.x(i + mol.nao_nr())

for orb_rot, (diag_mat_aa, diag_mat_ab) in zip(orbital_rotations, diag_mats):
    orbital_rotation(qc_lucj_raw, orb_rot.T.conj(), N)
    J_op(qc_lucj_raw, diag_mat_aa, diag_mat_ab, diag_mat_aa, time=-1, norb=mol.nao_nr())
    orbital_rotation(qc_lucj_raw, orb_rot, N)

if ucj_op.final_orbital_rotation is not None:
    orbital_rotation(qc_lucj_raw, ucj_op.final_orbital_rotation, N)

from qiskit import qasm2
qasm2.dump(qc_lucj_raw, "ansatz_circuit.qasm")

qc_lucj = pass_manager.run(qc_lucj)